### Langkah 3: Membagi Data (Train-Test Split)

Notebook ini berfokus pada pembagian dataset menjadi dua bagian:
1. Training Set (80% / 800 nasabah):
   - Bahan belajar bagi algoritma Machine Learning untuk menemukan pola kelayakan kredit.
2. Testing Set (20% / 200 nasabah):
   - Soal ujian yang tidak pernah dilihat oleh model sebelumnya, untuk menguji apakah model benar-benar cerdas atau hanya menghafal (*overfitting*).

### Catatan Penting: Stratified Split
Karena nasabah layak (70%) dan tidak layak (30%) tidak seimbang (70:30), kita menggunakan parameter stratify=y agar di data latih maupun data uji, proporsinya tetap tepat 70% layak dan 30% tidak layak.

### 1. Import Library & Load Data Hasil Preprocessing

In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# Deteksi otomatis lokasi file preprocessed_data.npz
possible_paths = [
    "preprocessed_data.npz",
    os.path.join("data", "preprocessed_data.npz"),
    os.path.join("..", "data", "preprocessed_data.npz")
]

DATA_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("File preprocessed_data.npz tidak ditemukan. Jalankan notebook 2_preprocessing.ipynb terlebih dahulu!")

# Memuat data numpy
data = np.load(DATA_PATH, allow_pickle=True)
X = data['X']
y = data['y']
feature_names = data['feature_names']

print(f"✓ Data berhasil dimuat dari: {DATA_PATH}")
print(f"✓ Total Data: {X.shape[0]} nasabah dengan {X.shape[1]} fitur input")
print(f"✓ Target y  : {len(y)} label")

✓ Data berhasil dimuat dari: data\preprocessed_data.npz
✓ Total Data: 1000 nasabah dengan 48 fitur input
✓ Target y  : 1000 label


### 2. Membagi Data Menjadi 80% Train dan 20% Test
- test_size=0.20: 20% untuk pengujian (200 nasabah).
- random_state=42: Kunci pengacakan agar hasil split konsisten dan dapat direproduksi ulang kapan saja.
- stratify=y: Menjaga rasio perbandingan kelas 1 (Layak) dan 0 (Tidak Layak) tetap identik di data latih dan data uji.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=== HASIL PEMBAGIAN DATA ===")
print(f"Jumlah Data Latih (X_train): {X_train.shape[0]} baris ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Jumlah Data Uji   (X_test) : {X_test.shape[0]} baris ({X_test.shape[0]/len(X)*100:.0f}%)")

=== HASIL PEMBAGIAN DATA ===
Jumlah Data Latih (X_train): 800 baris (80%)
Jumlah Data Uji   (X_test) : 200 baris (20%)


### 3. Verifikasi Proporsi Target (Stratification Check)
Mari kita pastikan proporsi nasabah Layak (1) dan Tidak Layak (0) tetap seimbang pada data latih dan data uji.

In [3]:
def print_distribution(labels, dataset_name):
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(unique, counts))
    total = len(labels)
    print(f"{dataset_name}:")
    print(f"  - Layak (1)      : {dist.get(1, 0)} nasabah ({dist.get(1, 0)/total*100:.1f}%)")
    print(f"  - Tidak Layak (0): {dist.get(0, 0)} nasabah ({dist.get(0, 0)/total*100:.1f}%)")

print_distribution(y_train, "Data Latih (Train)")
print()
print_distribution(y_test, "Data Uji (Test)")

Data Latih (Train):
  - Layak (1)      : 560 nasabah (70.0%)
  - Tidak Layak (0): 240 nasabah (30.0%)

Data Uji (Test):
  - Layak (1)      : 140 nasabah (70.0%)
  - Tidak Layak (0): 60 nasabah (30.0%)


### 4. Simpan Hasil Split Data
Kita simpan file split ini ke dalam train_test_data.npz agar notebook training selanjutnya tinggal memuat file ini langsung.

In [4]:
OUTPUT_DIR = os.path.dirname(DATA_PATH) if os.path.dirname(DATA_PATH) != "" else "."
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "train_test_data.npz")

np.savez_compressed(
    OUTPUT_FILE,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_names=feature_names
)

print(f"✓ Data Train-Test berhasil disimpan di: {OUTPUT_FILE}")
print("\nLangkah 3 selesai! Siap masuk ke Langkah 4: Melatih Model (Training).")

✓ Data Train-Test berhasil disimpan di: data\train_test_data.npz

Langkah 3 selesai! Siap masuk ke Langkah 4: Melatih Model (Training).
